In [7]:
import pandas as pd
from transformers import (
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    T5ForConditionalGeneration
)

In [8]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [9]:
train_data.shape


(14732, 3)

In [10]:
val_data.shape

(818, 3)

In [5]:
# Random Sampling
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)

# Data Pre_Processing

In [6]:
import re
def clean_data(text):
    text=re.sub(r"\r\n"," ",text) # line
    text=re.sub(r"\s+"," ",text)  #spaces
    text=re.sub(r"<.*?>"," ",text) # html tags <p1> <h1>
    text=text.strip().lower()
    return text   
    

In [7]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=train_data["dialogue"].apply(clean_data)
val_data["summary"]=train_data["summary"].apply(clean_data)

# Tokenize

In [8]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [9]:
# raw data => tokenize inputs for fine-tunning
def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncate=True)

    inputs["labels"]=targets["input_ids"] # token ids => add to input as labels
    return inputs

In [10]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [11]:
# input Ids- dialogue=> token ids
# 1 => EOS, 0=> padding
# attention mask
#labels-target=>summary token

# Working with our model

In [12]:
# NLP => generation task
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1676.95it/s]


In [13]:
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")
print(f"device: {device}")
model.to(device)
    

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:
# trainign Arguements
training_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0=> lr default
)

In [15]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [16]:
# trainer.train()

In [17]:
#model load => fine-tune => save the model

In [18]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.31s/it]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [19]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 19018.82it/s]


In [20]:
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue) # clean

  #tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  )

  # generate the summary ==> token ids
  model.to(device)
  targets=model.generate(
      input_ids=inputs["input_ids"].to(device),
      attention_mask=inputs["attention_mask"].to(device),
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # decode our output
  summary=tokenizer.decode(targets[0],skip_special_tokens=True) #EOS,SEP
  return summary

In [21]:
test_dialogue=""" **Rahul:** Hey Priya, have you noticed how often we use AI these days without even realizing it?

**Priya:** Absolutely! I use voice assistants to set reminders, and my phone's camera even uses AI to improve photos.

**Rahul:** Same here. Even when I watch videos or shop online, the recommendations seem to know exactly what I like.

**Priya:** That's because AI analyzes our preferences and behavior. It makes things more personalized.

**Rahul:** Do you think AI will replace human jobs?

**Priya:** I think it will automate some repetitive tasks, but it will also create new opportunities. People will need to learn new skills.

**Rahul:** That's a good point. I've started learning programming and machine learning basics for that reason.

**Priya:** Great idea! Understanding technology can really help in today's job market.

**Rahul:** Are there any downsides to AI?

**Priya:** Sure. Privacy concerns, misinformation, and overdependence on technology are some of the biggest challenges.

**Rahul:** So, the key is to use AI responsibly rather than fear it.

**Priya:** Exactly. When used ethically, AI can improve healthcare, education, transportation, and many other areas.

**Rahul:** Looks like the future will involve humans and AI working together instead of competing.

**Priya:** I agree. The best results will come from combining human creativity with AI's speed and efficiency.
"""

summary=summarize_dialogue(test_dialogue)
print("Summary: ",summary)

Summary:  ai can improve healthcare, education, transportation, and many other areas. ai can improve healthcare, education, transportation, and many other areas. when used ethically, ai can improve healthcare, education, transportation, and many other areas.
